# BFMN — Rassereno

Building of **behavioural forma mentis networks (BFMN)** for every sample, extraction of
their network features and export of the aggregated results.

In [1]:
import pandas as pd
import pickle

from config import RESSERENO_ROOT, KEYWORDS_RASSERENO, KEYWORDS_RASSERENO_ENG
from bfmn_building import (
    categorize_valence_multiple,
    pipeline_df_building,
    build_multiple_fmnt,
)

In [2]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

### Function defaults

`build_multiple_fmnt`:
- `flower="yes"` &nbsp;&rarr;&nbsp; plot the emotional flowers (set `"no"` to skip them)
- `language="ita"` &nbsp;&rarr;&nbsp; language EmoAtlas works in (set `"eng"` for English)

`pipeline_df_building`:
- `filter="no"` &nbsp;&rarr;&nbsp; keep every association (set `"yes"` to drop idiosyncratic ones, i.e. associations made by a single participant)

## Files reading

In [3]:
def read_formatting(rel_path):
    return pd.read_csv(RESSERENO_ROOT / "out" / "formatting" / rel_path)


def read_split(rel_path):
    return pd.read_csv(RESSERENO_ROOT / "out" / "splitted_dfs" / rel_path)


# already long-format association files
experts_2019 = read_formatting("experts_2019/associations_valence_experts_2019.csv")
students_2019 = read_formatting("students_2019/associations_valence_students_2019.csv")
physics_2025 = read_formatting("physics_2025/associations_valence_fisica.csv")
physics_gpt_oss = read_formatting("gpt_oss/associations_valence_physics.csv")

# anxiety-split raw files
psychology_high_anx = read_split("all_psychology_students/high_anx.csv")
psychology_low_anx = read_split("all_psychology_students/low_anx.csv")
gpt_oss_psychology_high_anx = read_split("psychology_gpt_oss/high_anx.csv")
gpt_oss_psychology_low_anx = read_split("psychology_gpt_oss/low_anx.csv")
high_school_2025_high_anx = read_split("high_school_2025/high_anx.csv")
high_school_2025_low_anx = read_split("high_school_2025/low_anx.csv")
gpt_oss_high_school_high_anx = read_split("high_school_gpt_oss/high_anx.csv")
gpt_oss_high_school_low_anx = read_split("high_school_gpt_oss/low_anx.csv")

## Dataframe preparation

The 2019 samples are already in long format, so they only need valence categorization.
Every other sample goes through the full `pipeline_df_building` (text cleaning &rarr; long format
&rarr; valence categorization).

In [4]:
# 2019 samples: valence categorization only
df_experts_2019 = categorize_valence_multiple(experts_2019)
df_high_school_2019 = categorize_valence_multiple(students_2019)

# normalise cue words / responses of the 2019 high-school data (lowercase, strip accents)
for col in ["cue_word", "word"]:
    if col in df_high_school_2019.columns:
        df_high_school_2019[col] = (
            df_high_school_2019[col].astype(str).str.lower().str.replace("à", "a")
        )

Number of unique words: 1470


Number of unique words: 4152


In [5]:
# full pipeline for the remaining samples
df_physics_students_2025 = pipeline_df_building(physics_2025)
df_gpt_oss_physics_2025 = pipeline_df_building(physics_gpt_oss)

df_high_school_2025_high_anx = pipeline_df_building(high_school_2025_high_anx)
df_high_school_2025_low_anx = pipeline_df_building(high_school_2025_low_anx)

df_gpt_oss_high_school_high_anx = pipeline_df_building(gpt_oss_high_school_high_anx)
df_gpt_oss_high_school_low_anx = pipeline_df_building(gpt_oss_high_school_low_anx)

df_gpt_oss_psychology_high_anx = pipeline_df_building(gpt_oss_psychology_high_anx)
df_gpt_oss_psychology_low_anx = pipeline_df_building(gpt_oss_psychology_low_anx)

# psychology students are prepared both filtered and unfiltered
df_psychology_high_anx_filtered = pipeline_df_building(
    psychology_high_anx, filter="yes"
)
df_psychology_high_anx_unfiltered = pipeline_df_building(
    psychology_high_anx, filter="no"
)
df_psychology_low_anx_filtered = pipeline_df_building(psychology_low_anx, filter="yes")
df_psychology_low_anx_unfiltered = pipeline_df_building(psychology_low_anx, filter="no")

Number of unique words: 1144
Number of unique words: 749
Number of unique words: 1207
Number of unique words: 1189
Number of unique words: 536
Number of unique words: 552
Number of unique words: 1025
Number of unique words: 1305
Number of unique words: 1069
Number of unique words: 3475
Number of unique words: 1137
Number of unique words: 3582


## BFMN building

Each sample is described by `(name, dataframe, keywords, output subfolder, extra kwargs)` and
built through the same `build_multiple_fmnt` call. Results are collected in `bfmn_results`, keyed
by sample name, each holding the keyword formae mentis, the per-keyword features and the whole-network
features.

In [ ]:
# (name, dataframe, keywords, output subfolder, extra build_multiple_fmnt kwargs)
BFMN_BUILDS = [
    (
        "experts",
        df_experts_2019,
        KEYWORDS_RASSERENO_ENG,
        "experts_2019",
        {"language": "eng"},
    ),
    ("high_school_2019", df_high_school_2019, KEYWORDS_RASSERENO, "liceo_2019", {}),
    (
        "high_school_2025_high_anx",
        df_high_school_2025_high_anx,
        KEYWORDS_RASSERENO,
        "liceo_2025/high_anxiety",
        {},
    ),
    (
        "high_school_2025_low_anx",
        df_high_school_2025_low_anx,
        KEYWORDS_RASSERENO,
        "liceo_2025/low_anxiety",
        {},
    ),
    (
        "gpt_high_school_high_anx",
        df_gpt_oss_high_school_high_anx,
        KEYWORDS_RASSERENO,
        "gpt_oss_high_school/high_anxiety",
        {},
    ),
    (
        "gpt_high_school_low_anx",
        df_gpt_oss_high_school_low_anx,
        KEYWORDS_RASSERENO,
        "gpt_oss_high_school/low_anxiety",
        {},
    ),
    (
        "physics",
        df_physics_students_2025,
        KEYWORDS_RASSERENO,
        "physics_students_2025",
        {},
    ),
    ("gpt_physics", df_gpt_oss_physics_2025, KEYWORDS_RASSERENO, "gpt_oss_physics", {}),
    (
        "psychology_high_anx_filtered",
        df_psychology_high_anx_filtered,
        KEYWORDS_RASSERENO,
        "psychology_students_filtered/high_anxiety",
        {"flower": "no"},
    ),
    (
        "psychology_high_anx_unfiltered",
        df_psychology_high_anx_unfiltered,
        KEYWORDS_RASSERENO,
        "psychology_students_unfiltered/high_anxiety",
        {"flower": "yes"},
    ),
    (
        "psychology_low_anx_filtered",
        df_psychology_low_anx_filtered,
        KEYWORDS_RASSERENO,
        "psychology_students_filtered/low_anxiety",
        {"flower": "no"},
    ),
    (
        "psychology_low_anx_unfiltered",
        df_psychology_low_anx_unfiltered,
        KEYWORDS_RASSERENO,
        "psychology_students_unfiltered/low_anxiety",
        {"flower": "yes"},
    ),
    (
        "gpt_psychology_high_anx",
        df_gpt_oss_psychology_high_anx,
        KEYWORDS_RASSERENO,
        "gpt_oss_psychology/high_anxiety",
        {},
    ),
    (
        "gpt_psychology_low_anx",
        df_gpt_oss_psychology_low_anx,
        KEYWORDS_RASSERENO,
        "gpt_oss_psychology/low_anxiety",
        {},
    ),
]

bfmn_results = {}
for name, df, keywords, file_path, kwargs in BFMN_BUILDS:
    fmnts, features_keyword, features_tot = build_multiple_fmnt(
        df, keywords, RESSERENO_ROOT, file_path, **kwargs
    )
    bfmn_results[name] = {
        "fmnts": fmnts,
        "features_keyword": features_keyword,
        "features_tot": features_tot,
    }

## BFMN exports

- `features_keyword_fmns.csv` / `features_tot_fmns.csv`: per-keyword and whole-network features for
  the reference samples (psychology in its **unfiltered** version).
- `all_bfmns.pkl`: the edge lists of every keyword forma mentis (psychology in its **filtered** version).

In [7]:
analysis_dir = RESSERENO_ROOT / "out" / "analysis"

# samples (and order) feeding the exported feature tables
FEATURE_EXPORT_ORDER = [
    "experts",
    "high_school_2019",
    "high_school_2025_high_anx",
    "high_school_2025_low_anx",
    "gpt_high_school_high_anx",
    "gpt_high_school_low_anx",
    "physics",
    "gpt_physics",
    "psychology_high_anx_unfiltered",
    "psychology_low_anx_unfiltered",
    "gpt_psychology_high_anx",
    "gpt_psychology_low_anx",
]

for feature_key, filename in [
    ("features_keyword", "features_keyword_fmns.csv"),
    ("features_tot", "features_tot_fmns.csv"),
]:
    combined = pd.concat(
        [bfmn_results[name][feature_key] for name in FEATURE_EXPORT_ORDER],
        ignore_index=True,
    )
    combined.to_csv(analysis_dir / filename, index=False)

In [8]:
# export key -> sample name (psychology exported in its filtered version)
BFMN_EXPORT = {
    "experts": "experts",
    "high_school_south": "high_school_2019",
    "high_school_high_anx": "high_school_2025_high_anx",
    "high_school_low_anx": "high_school_2025_low_anx",
    "gpt_oss_high_school_high_anx": "gpt_high_school_high_anx",
    "gpt_oss_high_school_low_anx": "gpt_high_school_low_anx",
    "physics": "physics",
    "gpt_oss_physics": "gpt_physics",
    "psychology_high_anx": "psychology_high_anx_filtered",
    "psychology_low_anx": "psychology_low_anx_filtered",
    "gpt_oss_psychology_high_anx": "gpt_psychology_high_anx",
    "gpt_oss_psychology_low_anx": "gpt_psychology_low_anx",
}

all_bfmns = {
    export_key: [fmnt.edges for fmnt in bfmn_results[name]["fmnts"]]
    for export_key, name in BFMN_EXPORT.items()
}

with open(analysis_dir / "all_bfmns.pkl", "wb") as f:
    pickle.dump(all_bfmns, f)